If the ingredient name contains "cream", "milk", etc. then say not lactose-free

If the ingredient name contains "meat", etc. not vegan/vegetarian

Contains "wheat", "flour", "pasta", etc. not gluten-free

In [1]:
import json

In [2]:
with open("./structured_ingredients.json", "r") as f:
    structured_ingredients = json.load(f)

In [3]:
supplemented_structured_ingredients = []

In [4]:
from data_supplementation_keywords import (
    LACTOSE_KEYWORDS,
    GLUTEN_KEYWORDS,
    NON_VEGETARIAN_KEYWORDS,
    NON_VEGAN_KEYWORDS,
)

In [5]:
def check_dietary_flag(name: str, keywords: list[str]) -> bool:
    """
    Returns False if any keyword is found in the ingredient name, else True (assumed compliancy)

    :param name: the ingredient name to check
    :type name: str
    :param keywords: the list of keywords to check for
    :type keywords: list[str]

    :return: False if any keyword is found, else True
    :rtype: bool
    """

    name_lower = name.lower()
    
    return not any(keyword in name_lower for keyword in keywords)

In [6]:
import copy
from tqdm.auto import tqdm

for ingredient in tqdm(structured_ingredients):
    supplemented_ingredient = copy.deepcopy(ingredient)

    name = ingredient["name"]
    nutritional_information = supplemented_ingredient["nutritional_information"]

    if nutritional_information["is_lactose_free"] is None:
        nutritional_information["is_lactose_free"] = check_dietary_flag(name, LACTOSE_KEYWORDS)
        
    if nutritional_information["is_gluten_free"] is None:
        nutritional_information["is_gluten_free"] = check_dietary_flag(name, GLUTEN_KEYWORDS)

    if nutritional_information["is_vegetarian"] is None:
        nutritional_information["is_vegetarian"] = check_dietary_flag(name, NON_VEGETARIAN_KEYWORDS)

    # if the ingredient is not vegetarian, it cannot be vegan. No need to check further
    if nutritional_information["is_vegetarian"] is False:
        nutritional_information["is_vegan"] = False
    elif nutritional_information["is_vegan"] is None:
        nutritional_information["is_vegan"] = check_dietary_flag(name, NON_VEGAN_KEYWORDS)

    supplemented_structured_ingredients.append(supplemented_ingredient)


  0%|          | 0/16185 [00:00<?, ?it/s]

In [11]:
# Get counts of all null values for nutritional information fields after supplementation
from collections import Counter

counter = Counter()

print(f"Total ingredients after supplementation: {len(supplemented_structured_ingredients)}")

print("Null value counts for nutritional information fields after supplementation:")
for field in supplemented_structured_ingredients[0]["nutritional_information"].keys():
    null_count = sum(1 for ingredient in supplemented_structured_ingredients if ingredient["nutritional_information"][field] is None)
    counter[field] = null_count
    print(f"- {field}: {null_count}")

Total ingredients after supplementation: 16185
Null value counts for nutritional information fields after supplementation:
- calories: 397
- carbohydrates: 429
- sugar: 2818
- protein: 327
- fat: 368
- saturated_fat: 2900
- fiber: 3046
- sodium: 462
- is_gluten_free: 0
- is_lactose_free: 0
- is_vegetarian: 0
- is_vegan: 0


In [8]:
json.dump(supplemented_structured_ingredients, open("./supplemented_structured_ingredients.json", "w"), indent = 4)